# Kerala Tourism Trend Outlier Analysis Using Outlier Detection
## Module 2: Local Outlier Factor (LOF) Implementation, Evaluation, Comparison & Visualization
**Author / Responsible Team Member:** Nifa  
**Project Team Roles:**
- **Gauri:** Data Preprocessing (`preprocessing.py` $\rightarrow$ `data/processed_tourism.csv`)
- **Melisa:** Primary Algorithm: Isolation Forest (`outlier_detection.py` $\rightarrow$ `results/isolation_forest_results.csv`)
- **Nifa:** Second Algorithm: Local Outlier Factor (LOF), Comparative Evaluation, Visualization (`src/evaluation.py`, `src/visualization.py`, `notebooks/nifa_evaluation.ipynb`)

---

### Methodological Framework & Important Guidelines
1. **Unsupervised Outlier Detection:**
   In this project, true anomaly labels are unobserved in the real-world tourism data. Therefore, **supervised metrics such as Accuracy, Precision, Recall, and F1-score are mathematically undefined and methodologically invalid**. Computing precision or recall by treating one unsupervised algorithm as ground truth is a severe methodological fallacy.
2. **Algorithm Disagreement:**
   Disagreement between Local Outlier Factor (LOF) and Isolation Forest (IF) does **not** imply that either algorithm is incorrect:
   - **Isolation Forest** isolates points through recursive random partitioning. It excels at identifying *global outliers* residing in low-density, extreme regions.
   - **Local Outlier Factor (LOF)** measures the local reachability density relative to $k$-nearest neighbors. It detects *local outliers*—points whose density is substantially lower than that of their surrounding cluster, even if they lie within moderate overall feature ranges.
   - Points flagged by **both** algorithms represent high-confidence consensus anomalies.
3. **Zero Data Fabrication:**
   Per project requirements, no mock or fake tourism records are invented. All analyses dynamically adapt to the actual cleaned dataset provided by Gauri, and comparison gracefully validates the availability of Melisa's output.

In [ ]:
# Environment setup and imports
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to sys.path to enable imports from src
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Also add current directory if running from root
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import src.evaluation as ev
import src.visualization as vz

print("[OK] All required libraries and Nifa's modules (src.evaluation, src.visualization) imported successfully.")

---
## 1. Cleaned Tourism Dataset Ingestion & Column Inspection

Gauri's module cleans raw Kerala tourism records (handling missing values, data types, and formatting) and writes to `data/processed_tourism.csv`.

The cell below verifies dataset existence, loads the file, and inspects the actual columns and data types.

In [ ]:
DATA_PATH = os.path.join(project_root, 'data', 'processed_tourism.csv')
if not os.path.exists(DATA_PATH):
    # Check relative path from root
    if os.path.exists('data/processed_tourism.csv'):
        DATA_PATH = 'data/processed_tourism.csv'

try:
    tourism_df = ev.load_processed_tourism_data(DATA_PATH)
    print("\n--- First 5 Records ---")
    display(tourism_df.head())
    print("\n--- Column Summary & Data Types ---")
    tourism_df.info()
except FileNotFoundError as err:
    print(err)
    tourism_df = None

---
## 2. Feature Selection & Preprocessing for LOF

### Why LOF Requires Specific Preprocessing:
1. **Scale Sensitivity in Distance Computations:**  
   Local Outlier Factor relies on Euclidean / Minkowski distances to find the $k$-nearest neighbors of each point. If features have vastly different scales (e.g., total tourist arrivals in hundreds of thousands vs. foreign visitor percentages in single digits), unscaled distances will be completely dominated by the high-magnitude feature, rendering other features irrelevant.  
   Therefore, **feature scaling (StandardScaler or RobustScaler)** is mathematically essential.
2. **Missing Value Imputation:**  
   Distance metrics cannot compute pairwise distances with missing (NaN) values. Missing values are imputed systematically (e.g., using median values) prior to neighborhood density estimation.

In [ ]:
if tourism_df is not None:
    # Automatically inspect and select numeric columns suitable for LOF
    selected_features = ev.inspect_and_select_features(tourism_df)
    print(f"\n[OK] Features selected for LOF: {selected_features}")
    
    # Apply missing value handling and standardization
    X_scaled, imputer, scaler = ev.preprocess_features_for_lof(
        tourism_df,
        feature_cols=selected_features,
        handle_missing='median',
        scaler_type='standard'
    )
    
    # Preview scaled feature summary
    scaled_summary_df = pd.DataFrame(X_scaled, columns=selected_features)
    display(scaled_summary_df.describe().round(3))
else:
    print("[STATUS] Waiting for Gauri to supply 'data/processed_tourism.csv' before feature extraction.")

---
## 3. Second-Algorithm Implementation: Local Outlier Factor (LOF)

### Mathematical Principle of LOF (Breunig et al., 2000):
- **$k$-distance of $p$ ($d_k(p)$):** Distance between $p$ and its $k$-th nearest neighbor.
- **Reachability Distance:** $\text{reach-dist}_k(p, o) = \max(d_k(o), d(p, o))$.
- **Local Reachability Density ($lrd_k(p)$):**  
  $$\text{lrd}_k(p) = \frac{|N_k(p)|}{\sum_{o \in N_k(p)} \text{reach-dist}_k(p, o)}$$
- **Local Outlier Factor ($LOF_k(p)$):**  
  $$\text{LOF}_k(p) = \frac{\sum_{o \in N_k(p)} \frac{\text{lrd}_k(o)}{\text{lrd}_k(p)}}{|N_k(p)|}$$

- An observation with $LOF \approx 1.0$ possesses density comparable to its neighbors (**inlier**).
- An observation with $LOF > 1.0$ (e.g., $> 1.5$) has substantially lower local density than its neighbors (**local outlier**).

### Hyperparameter Configuration:
- `n_neighbors`: Number of neighbors ($k$) for local density estimation. Default: 20.
- `contamination`: Expected proportion of anomalies in the dataset. Default: 0.05 (5%).

In [ ]:
# Configurable LOF Hyperparameters
N_NEIGHBORS = 20
CONTAMINATION = 0.05
LOF_OUTPUT_PATH = os.path.join(project_root, 'results', 'lof_results.csv')

if tourism_df is not None and 'X_scaled' in locals():
    lof_model, is_outlier, raw_pred, lof_scores = ev.fit_local_outlier_factor(
        X_scaled,
        n_neighbors=N_NEIGHBORS,
        contamination=CONTAMINATION
    )
    
    # Create enriched results dataframe
    lof_results_df = tourism_df.copy()
    lof_results_df['lof_prediction'] = raw_pred
    lof_results_df['is_lof_outlier'] = is_outlier
    lof_results_df['negative_outlier_factor'] = lof_model.negative_outlier_factor_
    lof_results_df['lof_score'] = lof_scores
    
    # Ensure results directory exists and save
    dir_name = os.path.dirname(LOF_OUTPUT_PATH)
    if dir_name:
        os.makedirs(dir_name, exist_ok=True)
    lof_results_df.to_csv(LOF_OUTPUT_PATH, index=False)
    print(f"\n[SAVED] LOF results saved successfully to: {LOF_OUTPUT_PATH}")
    
    # Display detected outliers
    outliers_detected = lof_results_df[lof_results_df['is_lof_outlier'] == 1]
    print(f"\n[PREVIEW] Displaying top {min(5, len(outliers_detected))} of {len(outliers_detected)} LOF outliers:")
    display(outliers_detected.head())
else:
    print("[STATUS] Cannot train LOF until input data is available.")

### LOF Anomaly Score Distribution Visualization

In [ ]:
if 'lof_results_df' in locals():
    fig_dist, ax_dist = vz.plot_lof_score_distribution(
        lof_results_df,
        score_col='lof_score',
        title='Kerala Tourism - LOF Anomaly Score Distribution',
        save_path=os.path.join(project_root, 'results', 'lof_score_distribution.png')
    )
    plt.show()
else:
    print("[STATUS] Waiting for LOF model training to plot score distribution.")

---
## 4. Isolation Forest Results Ingestion & Availability Check

Melisa is responsible for developing the Isolation Forest model in `src/outlier_detection.py` and outputting `results/isolation_forest_results.csv`.

> **Zero-Fabrication Guarantee:** If Melisa's output file is not available, the system will explicitly report the missing prerequisite rather than fabricating fake comparison numbers.

In [ ]:
IF_RESULTS_PATH = os.path.join(project_root, 'results', 'isolation_forest_results.csv')
if not os.path.exists(IF_RESULTS_PATH):
    if os.path.exists('results/isolation_forest_results.csv'):
        IF_RESULTS_PATH = 'results/isolation_forest_results.csv'

try:
    if_results_df = ev.load_isolation_forest_results(IF_RESULTS_PATH)
    print("\n--- Melisa's Isolation Forest Results Preview ---")
    display(if_results_df.head())
except FileNotFoundError as err:
    print(err)
    if_results_df = None

---
## 5. Comparative Evaluation: Local Outlier Factor (LOF) vs. Isolation Forest

### Unsupervised Evaluation Principles:
- **No Ground-Truth Anomaly Labels:** In the absence of confirmed ground truth, accuracy, precision, recall, and F1-score are **not defined**.
- **Meaning of Algorithm Disagreement:** Disagreement is expected and valuable. Isolation Forest detects global anomalies in sparse subspaces, whereas LOF identifies points that deviate from their local cluster density. Both perspectives provide unique domain insights into tourism trends.

### Evaluated Comparative Metrics:
1. **Outlier Count & Detection Rate:** Percentage of records flagged by each method.
2. **Consensus Outliers (Both):** Points identified by both algorithms (high confidence).
3. **Algorithm-Specific Outliers:** Points flagged exclusively by LOF or exclusively by Isolation Forest.
4. **Agreed Inliers:** Points considered normal by both algorithms.
5. **Jaccard Similarity Index on Outliers:**
   $$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$
6. **Overall Concordance Rate:** Total agreement percentage across all data points.

In [ ]:
if 'lof_results_df' in locals() and if_results_df is not None:
    # Perform comparison
    merged_comparison_df, comparison_summary = ev.compare_outlier_algorithms(
        lof_results_df,
        if_results_df
    )
    
    # Display formatted metrics table
    summary_table = pd.DataFrame([
        {"Metric": "Total Records", "Count": comparison_summary["total_records"], "Percentage (%)": 100.0},
        {"Metric": "LOF Outliers Detected", "Count": comparison_summary["lof_outliers_count"], "Percentage (%)": comparison_summary["lof_outliers_pct"]},
        {"Metric": "Isolation Forest Outliers Detected", "Count": comparison_summary["if_outliers_count"], "Percentage (%)": comparison_summary["if_outliers_pct"]},
        {"Metric": "Mutual Consensus Outliers (Both)", "Count": comparison_summary["both_outliers_count"], "Percentage (%)": comparison_summary["both_outliers_pct"]},
        {"Metric": "LOF-Only Detections (Local Density Outliers)", "Count": comparison_summary["lof_only_count"], "Percentage (%)": comparison_summary["lof_only_pct"]},
        {"Metric": "Isolation Forest-Only Detections (Global Outliers)", "Count": comparison_summary["if_only_count"], "Percentage (%)": comparison_summary["if_only_pct"]},
        {"Metric": "Agreed Inliers (Neither)", "Count": comparison_summary["inliers_count"], "Percentage (%)": comparison_summary["inliers_pct"]},
    ])
    
    print("\n--- Summary Table ---")
    display(summary_table)
    
    print(f"\n[Jaccard Similarity Index on Outlier Sets]: {comparison_summary['jaccard_similarity']:.4f}")
    print(f"[Overall Inlier/Outlier Concordance Rate]: {comparison_summary['concordance_rate_pct']:.2f}%")
    print(f"[LOF Outliers confirmed by IF]:           {comparison_summary['overlap_percentage_of_lof']:.2f}%")
    print(f"[IF Outliers confirmed by LOF]:            {comparison_summary['overlap_percentage_of_if']:.2f}%")
else:
    print("[NOTICE] Comparison pending availability of both 'data/processed_tourism.csv' and 'results/isolation_forest_results.csv'.")

---
## 6. Comparison Visualizations

Visualizations generated by `src/visualization.py`:
1. **Summary Bar Charts:** Overall detection rates and consensus category breakdown.
2. **2D Feature Scatter Comparison:** Multi-algorithm consensus across key tourism dimensions.
3. **Temporal Outlier Trend:** Chronological evolution of tourism volume highlighting seasonal spikes and unexpected dips.

In [ ]:
if 'comparison_summary' in locals() and 'merged_comparison_df' in locals():
    # 1. Summary comparison plot
    fig_comp, _ = vz.plot_comparison_summary(
        comparison_summary,
        save_path=os.path.join(project_root, 'results', 'outlier_comparison_summary.png')
    )
    plt.show()
    
    # 2. 2D Feature Scatter Comparison (pick top 2 numeric features if present)
    if len(selected_features) >= 2:
        f1, f2 = selected_features[0], selected_features[1]
        fig_scat, _ = vz.plot_feature_scatter_comparison(
            merged_comparison_df,
            x_col=f1,
            y_col=f2,
            save_path=os.path.join(project_root, 'results', 'feature_scatter_comparison.png')
        )
        plt.show()
        
    # 3. Temporal Trend Outlier Analysis (if date/month column exists)
    date_col = next((c for c in ['date', 'month', 'year', 'month_year', 'period'] if c in merged_comparison_df.columns), None)
    metric_col = selected_features[0] if selected_features else None
    
    if date_col and metric_col:
        fig_trend, _ = vz.plot_temporal_outlier_trend(
            merged_comparison_df,
            date_or_index_col=date_col,
            metric_col=metric_col,
            save_path=os.path.join(project_root, 'results', 'temporal_outlier_trend.png')
        )
        plt.show()
else:
    print("[NOTICE] Comparison visualizations will render automatically once both models have executed.")

---
## 7. Group Integration Guide (`main.py`) & Final Report Instructions

### How the Group Can Integrate Nifa's Code into `main.py`:
```python
# ==========================================================
# Example snippet for main.py
# ==========================================================
from src.preprocessing import clean_tourism_data          # Gauri
from src.outlier_detection import run_isolation_forest     # Melisa
from src.evaluation import run_lof_detection, compare_outlier_algorithms # Nifa
from src.visualization import plot_comparison_summary, plot_feature_scatter_comparison # Nifa

# Step 1: Preprocessing (Gauri)
# cleaned_df = clean_tourism_data('data/raw_tourism.csv', 'data/processed_tourism.csv')

# Step 2: Isolation Forest (Melisa)
# if_results = run_isolation_forest('data/processed_tourism.csv', 'results/isolation_forest_results.csv')

# Step 3: Local Outlier Factor (Nifa)
lof_results = run_lof_detection(
    data_path='data/processed_tourism.csv',
    output_path='results/lof_results.csv',
    n_neighbors=20,
    contamination=0.05
)

# Step 4: Comparison & Evaluation (Nifa)
merged_results, summary = compare_outlier_algorithms(lof_results, if_results)

# Step 5: Visualizations (Nifa)
plot_comparison_summary(summary, save_path='results/comparison_summary.png')
plot_feature_scatter_comparison(merged_results, x_col='domestic_tourists', y_col='foreign_tourists', save_path='results/feature_scatter.png')
```

### How to Use Comparison Results in the Final Report:
1. **Consensus Outliers as Verified Anomalies:**  
   Highlight the records detected by **both** Isolation Forest and LOF as primary case studies for the report (e.g. statewide disruptions like Kerala floods in August 2018 or massive festival surges during Onam).
2. **Algorithm Differentiation Analysis:**  
   Explain why points flagged by only LOF represent local density drops (e.g. a specific district experiencing an abnormal drop relative to other similar districts in the same monsoon month), whereas points flagged only by Isolation Forest represent global scale extremes.
3. **Methodological Rigor in Evaluation:**  
   Explicitly emphasize in the report that accuracy/precision/recall are inappropriate for unsupervised outlier analysis, and use the Jaccard similarity index and concordance rate to describe model alignment.